In [ ]:

"""
================================================================================
MIXPANEL EVENT EXPORTER
================================================================================

A simple tool to export raw Mixpanel events to CSV via the Export API.

HOW TO USE:
-----------
1. Run this cell in a Jupyter notebook or similar python tool.
2. Enter your Mixpanel credentials when prompted:
   - Project ID (found in Mixpanel → Settings → Project Settings)
   - Service Account Username & Secret (create one in Mixpanel → Settings →
     Service Accounts → Add Service Account)
3. Enter the date range (YYYY-MM-DD format).
4. Choose whether to export all events or filter specific ones.
5. The tool fetches data in weekly chunks and saves a CSV in your notebook's
   working directory.

CAVEATS:
--------
- The Mixpanel Export API is only available on the Growth plan or above.
  It will not work on the Free plan.

- The Export API can return higher event counts than what you see in
  Mixpanel's UI. This is because the API exports all raw data, including
  events that Mixpanel may not have ingested due to deduplication. You will
  need to deduplicate the exported data yourself (e.g. using the
  $insert_id property).

================================================================================
"""

import base64
import json
import csv
import requests
from urllib.parse import quote_plus
from datetime import datetime, timedelta, timezone
from getpass import getpass

# ── User Inputs ──────────────────────────────────────────────
project_id = input("Mixpanel Project ID: ").strip()
username = input("Service Account Username: ").strip()
secret = getpass("Service Account Secret: ").strip()
from_date = input("From Date (YYYY-MM-DD): ").strip()
to_date = input("To Date (YYYY-MM-DD): ").strip()

filter_choice = input("Filter specific events? (y/n): ").strip().lower()
events = None
if filter_choice == "y":
    raw = input("Event names (comma-separated): ").strip()
    events = [e.strip() for e in raw.split(",") if e.strip()]

# ── Helpers ──────────────────────────────────────────────────
EXPORT_URL = "https://data.mixpanel.com/api/2.0/export"
auth = base64.b64encode(f"{username}:{secret}".encode()).decode()

def flatten(obj, prefix=""):
    out = {}
    for k, v in (obj or {}).items():
        key = f"{prefix}.{k}" if prefix else k
        if isinstance(v, dict):
            out.update(flatten(v, key))
        elif isinstance(v, list):
            out[key] = json.dumps(v)
        else:
            out[key] = v
    return out

def epoch_to_rfc3339(sec):
    try:
        return datetime.fromtimestamp(int(float(sec)), tz=timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    except Exception:
        return sec

def fetch_week(week_start, week_end):
    """Fetch one week chunk from Mixpanel."""
    fd = week_start.strftime("%Y-%m-%d")
    td = week_end.strftime("%Y-%m-%d")
    params = f"project_id={project_id}&from_date={fd}&to_date={td}"
    if events:
        params += "&event=" + quote_plus(json.dumps(events))

    r = requests.get(
        f"{EXPORT_URL}?{params}",
        headers={"Authorization": f"Basic {auth}", "accept": "text/plain"},
        stream=True,
        timeout=300,
    )
    if r.status_code != 200:
        raise RuntimeError(f"{fd} to {td}: Export error {r.status_code} -> {r.text[:300]}")

    chunk = []
    for line in r.iter_lines():
        if not line or not line.strip():
            continue
        text = line.decode("utf-8", errors="replace").strip()
        if not text:
            continue
        if text.lower() == "terminated early":
            raise RuntimeError(f"{fd} to {td}: Stream terminated early. Try smaller ranges.")
        chunk.append(json.loads(text))
    return chunk

# ── Build weekly chunks ──────────────────────────────────────
start = datetime.strptime(from_date, "%Y-%m-%d")
end = datetime.strptime(to_date, "%Y-%m-%d")

weeks = []
cur = start
while cur <= end:
    week_end = min(cur + timedelta(days=6), end)
    weeks.append((cur, week_end))
    cur = week_end + timedelta(days=1)

print(f"\nFetching {from_date} to {to_date} in {len(weeks)} week(s)...\n")

# ── Fetch all weeks ──────────────────────────────────────────
rows = []
all_keys = set()

for i, (ws, we) in enumerate(weeks, 1):
    label = f"{ws.strftime('%Y-%m-%d')} to {we.strftime('%Y-%m-%d')}"
    print(f"[{i}/{len(weeks)}] Fetching {label}...", end=" ")

    raw_events = fetch_week(ws, we)

    for evt in raw_events:
        flat = {"event": evt.get("event")}
        flat.update(flatten(evt.get("properties", {})))
        if "time" in flat:
            flat["time"] = epoch_to_rfc3339(flat["time"])
        all_keys.update(flat.keys())
        rows.append(flat)

    print(f"{len(raw_events):,} events")

print(f"\nTotal: {len(rows):,} events with {len(all_keys)} columns.")

# ── Write CSV ────────────────────────────────────────────────
cols = ["event"] + sorted(k for k in all_keys if k != "event")
filename = f"mixpanel_export_{from_date}_to_{to_date}.csv"

with open(filename, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
    writer.writeheader()
    for row in rows:
        writer.writerow(row)

print(f"\nSaved to {filename}")